In [0]:
# All the imports
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
# ------------------------------------------------------------
# 1. Parameter: initial vs incremental load
#    1 = very first load (dimension empty/doesn't exist)
#    0 = incremental load (dimension already has data)
# ------------------------------------------------------------

init_load_flag = int(dbutils.widgets.get("initial_load_flag"))

In [0]:
# ------------------------------------------------------------
# 2. Read the cleaned silver customers
# ------------------------------------------------------------
# Using spark.sql("sql query .."). ?
df = spark.sql("select * from databricks_cata.silver.customers_silver")
df.display()

In [0]:
# 3. Geting the OLD records (existing dimension) — the flag branch
#    Incremental: read existing DimCustomers keys
#    Initial:     build a TYPED EMPTY placeholder (WHERE 1=0)
#    so downstream join/split code works identically either way

if init_load_flag == 0:
    df_old = spark.sql('''
                       SELECT DimCustomerKey,
                       customer_id,
                       create_date,
                       update_date
                       FROM databricks_cata.gold.DimCustomers
                       ''')
else:
    """
    df_old is for: it represents what's already in the gold dimension, and its only job is to answer "does this incoming customer already exist, and if so, what's their existing key and history?" We don't need the old attributes (name, city, etc.) because SCD Type 1 overwrites — the new values from silver win anyway. So we only pull the columns the merge logic actually consumes:
    """
    df_old = spark.sql('''
                       SELECT 0 as DimcustomerKey,
                       0 as customer_id,
                       0 as create_date,
                       0 as update_date
                       From databricks_cata.silver.customers_silver
                       WHERE 1=0
    
                       ''')
df_old.display()

In [0]:

# 4. Rename df_old columns with old_ prefix
#    After the join both sides have customer_id etc. — prefix
#    avoids ambiguous column references.

df_old = df_old.withColumnRenamed("DimcustomerKey","old_DimcustomerKey")\
    .withColumnRenamed("customer_id","old_customer_id")\
    .withColumnRenamed("create_date","old_create_date")\
    .withColumnRenamed("update_date","old_update_date")
df_old.display()


In [0]:
# 5. LEFT JOIN incoming silver against existing dimension
#    Match on business key (customer_id).
#    Matched   -> old_DimCustomerKey populated -> UPDATE
#    No match  -> old_DimCustomerKey is NULL   -> INSERT

df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, "left")
df_join.display()

In [0]:
# 6. Split into new (inserts) vs old (updates)
df_new = df_join.filter(col("old_DimcustomerKey").isNull())
# Since we are doing SCD TYPE 1, I will not consider any old kept data , i JUST NEED THE NEWLY Updated data which is currently present inthe JOINED DF df_join
df_upd = df_join.filter(col("old_DimcustomerKey").isNotNull())

#df_new.display()
df_upd.display()

In [0]:
# 7. PREPARING the UPDATES (df_upd)
#    - keep original surrogate key
#    - preserve original create_date
#    - refresh update_date to now
#    - drop the old_ helper columns we no longer need

#withColumn can also be used to Renaming like task, in the below code it will create a new column with name as DimCustomerKey and value as old_DimCustomerKey, But then i need to remove the old_DimCustomerKey column

df_upd = df_upd.withColumn("DimCustomerKey", col("old_DimcustomerKey"))\
    .withColumn("create_date", to_timestamp(col("old_create_date")))\
        .withColumn("update_Date", current_timestamp()) 

#now dropping the old HELPER columns
df_upd = df_upd.drop("old_DimcustomerKey","old_create_date","old_customer_id","old_update_date")

df_upd.display()


##### 8. PREPARING the NEW INSERTS (df_new)

In [0]:
# 8. PREPARING the NEW INSERTS (df_new)
#    - generate brand-new surrogate keys
#    - keys must CONTINUE from the current max in the dimension
#    - both create_date and update_date = now

df_new = df_new.withColumn("create_date", current_timestamp())\
    .withColumn("update_date", current_timestamp())

df_new = df_new.drop("old_DimcustomerKey","old_customer_id","old_create_date","old_update_date")

#new surrogate keys max of the surrogate key + 1 -> running 
#finding the correct surrogate key



###### getting the correct surrogate key to start 

In [0]:
#new surrogate keys max of the surrogate key + 1 -> running 
#finding the correct surrogate key

if init_load_flag == 0:
    max_key = spark.sql('''
                        Select max(DimCustomerKey) as max_key
                        from databricks_cata.gold.DimCustomers
                        ''').collect()[0][0]
    max_key = max_key if max_key is not None else 0

    """.collect() pulls the DataFrame's rows out of Spark and back to the driver as a Python listof Row objects. For a MAX() query that's a list with exactly one row
        [0] — grab the first (only) row from that list → Row(max(DimCustomerKey)=1900).

    [0] again — grab the first (only) column's value from that row → 1900"""
    #MAX() on an empty table returns NULL None + row_number() would crash
else:
    max_key = 0


#assigning new sequential keys = max_key + row_number()
#Since we are dealing with the new data only, which are definitely not having any value in DimCustomerKey, so we can directly assign the new surrogate key as max_key + row_number()
window_spec = Window.orderBy("customer_id")
df_new = df_new.withColumn("DimCustomerKey", lit(max_key) + row_number().over(window_spec))
#lit() wraps the Python value into a Spark column literal so Spark can combine them



    

In [0]:
df_new.display()

In [0]:
# 9. UNION updates + inserts into the final dimension frame
#    (align column order so union is safe)

df_final = df_new.unionByName(df_upd)


In [0]:
df_final.display()


In [0]:
# 10. WRITE to gold DimCustomers (Delta)
#     Initial load -> overwrite (create the table)
#     Incremental  -> overwrite the whole dimension with the
#                     recomputed set (simple, tutorial approach)

if spark.catalog.tableExists("databricks_cata.gold.DimCustomers"):
    #table already exists. this is a incremental load
    delta_obj = DeltaTable.forPath(spark, "abfss://gold@monarchazuredatalake.dfs.core.windows.net/DimCustomers")

    delta_obj.alias("target").merge(df_final.alias("source"), "target.DimCustomerKey = source.DimCustomerKey")\
        .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
                .execute()
else:
    #Tbale does not exisit yet -> first lets load and -> plain overright to create it
    df_final.write.mode("overwrite").format("delta")\
        .option("path", "abfss://gold@monarchazuredatalake.dfs.core.windows.net/DimCustomers")\
            .saveAsTable("databricks_cata.gold.DimCustomers")


        